# Quality-weight three-seed report

Upload or locate `quality_weight_3seed_artifacts.zip`. This notebook performs no training or inference and does not need the dataset.

## Step 1 — Load and validate the artifact


In [1]:
import csv,io,json,zipfile,sys,statistics
from pathlib import Path
from IPython.display import display,Markdown
NAME="quality_weight_3seed_artifacts.zip"
paths=[Path(NAME),Path("/content")/NAME,Path("/content/drive/MyDrive/CITS4012_A2_quality_confirm")/NAME]
archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import drive; drive.mount("/content/drive"); archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import files
    up=files.upload(); assert NAME in up; archive=Path("/content")/NAME; archive.write_bytes(up[NAME])
assert archive is not None
zf=zipfile.ZipFile(archive); assert zf.testzip() is None
summary=json.loads(zf.read("quality_results/quality_summary.json")); print("Loaded",archive,"— no training or inference")


Loaded quality_weight_3seed_artifacts.zip — no training or inference


## Step 2 — Compare three-seed results


In [2]:
metrics=("bleu1","meteor","rougeL","em","f1")
lines=["| Model | Seed | BLEU-1 | METEOR | ROUGE-L | EM | F1 |","|---|---:|---:|---:|---:|---:|---:|"]
for name in ("ObjectiveControl","QualityWeight"):
    for seed in summary["seeds"]:
        v=summary["runs"][name][str(seed)]
        lines.append(f"| {name} | {seed} | "+" | ".join(f"{v[m]:.2f}" for m in metrics)+" |")
    a=summary["aggregate"][name]
    lines.append(f"| **{name} mean ± SD** | — | "+" | ".join(f"**{a[m]['mean']:.2f} ± {a[m]['std']:.2f}**" for m in metrics)+" |")
display(Markdown("## Three-seed results\n\n"+"\n".join(lines)))
d=summary["delta_quality_minus_control"]
display(Markdown("## QualityWeight − Control\n\n"+"\n".join([
    "| Metric | Seed 1 | Seed 2 | Seed 3 | Mean ± SD |","|---|---:|---:|---:|---:|",
    *[f"| {m} | "+" | ".join(f"{x:+.2f}" for x in d[m]["by_seed"])+f" | **{d[m]['mean']:+.2f} ± {d[m]['std']:.2f}** |" for m in metrics]])))


## Three-seed results

| Model | Seed | BLEU-1 | METEOR | ROUGE-L | EM | F1 |
|---|---:|---:|---:|---:|---:|---:|
| ObjectiveControl | 1 | 38.62 | 32.99 | 43.88 | 27.23 | 43.95 |
| ObjectiveControl | 2 | 38.84 | 32.50 | 44.11 | 27.79 | 44.33 |
| ObjectiveControl | 3 | 38.56 | 32.62 | 44.01 | 27.14 | 44.16 |
| **ObjectiveControl mean ± SD** | — | **38.68 ± 0.15** | **32.70 ± 0.26** | **44.00 ± 0.12** | **27.39 ± 0.35** | **44.15 ± 0.19** |
| QualityWeight | 1 | 39.50 | 33.36 | 45.02 | 28.44 | 45.25 |
| QualityWeight | 2 | 38.17 | 32.77 | 43.43 | 27.42 | 43.56 |
| QualityWeight | 3 | 39.36 | 33.76 | 44.87 | 27.88 | 44.97 |
| **QualityWeight mean ± SD** | — | **39.01 ± 0.73** | **33.29 ± 0.50** | **44.44 ± 0.88** | **27.91 ± 0.51** | **44.59 ± 0.91** |

## QualityWeight − Control

| Metric | Seed 1 | Seed 2 | Seed 3 | Mean ± SD |
|---|---:|---:|---:|---:|
| bleu1 | +0.87 | -0.68 | +0.80 | **+0.33 ± 0.88** |
| meteor | +0.37 | +0.26 | +1.14 | **+0.59 ± 0.48** |
| rougeL | +1.15 | -0.68 | +0.86 | **+0.44 ± 0.98** |
| em | +1.21 | -0.37 | +0.74 | **+0.53 ± 0.81** |
| f1 | +1.30 | -0.77 | +0.81 | **+0.45 ± 1.08** |

## Step 3 — Run paired bootstrap analysis


In [3]:
import numpy as np
rows=list(csv.DictReader(io.StringIO(zf.read("quality_results/paired_deltas.csv").decode())))
rng=np.random.default_rng(4012); lines=["| Metric | Mean paired Δ | 95% bootstrap CI | P(Δ≤0) |","|---|---:|---:|---:|"]
for metric in ("rougeL","f1","em"):
    # Cluster bootstrap: sample validation questions, keeping their three seed results together.
    grouped={}
    for r in rows: grouped.setdefault(r["row_id"],[]).append(float(r[metric]))
    values=np.array([grouped[k] for k in sorted(grouped)],dtype=float)
    means=np.array([values[rng.integers(0,len(values),len(values))].mean() for _ in range(10000)])*100
    lo,hi=np.quantile(means,[.025,.975]); point=values.mean()*100
    lines.append(f"| {metric} | {point:+.2f} | [{lo:+.2f}, {hi:+.2f}] | {(means<=0).mean():.3f} |")
display(Markdown("## Paired validation comparison\n\n"+"\n".join(lines)+"\n\nThe bootstrap resamples validation questions and keeps all three seed results for each question together."))


## Paired validation comparison

| Metric | Mean paired Δ | 95% bootstrap CI | P(Δ≤0) |
|---|---:|---:|---:|
| rougeL | +0.44 | [-0.41, +1.28] | 0.157 |
| f1 | +0.45 | [-0.41, +1.29] | 0.153 |
| em | +0.53 | [-0.37, +1.39] | 0.137 |

The bootstrap resamples validation questions and keeps all three seed results for each question together.

## Step 4 — Write the conclusion


In [4]:
r=d["rougeL"]; f=d["f1"]
decision="supported" if r["mean"]>0 and all(x>0 for x in r["by_seed"]) else "not yet supported"
text=f"""# Quality-weight confirmation

Across three matched seeds, quality weighting changed ROUGE-L by **{r['mean']:+.2f} ± {r['std']:.2f}**
and F1 by **{f['mean']:+.2f} ± {f['std']:.2f}** relative to the unchanged objective.
The improvement is **{decision}** by the predeclared practical rule: positive mean and positive ROUGE-L
effect in every seed. Results are validation-only; the locked test set was not accessed.
"""
Path("quality_weight_3seed_report.md").write_text(text,encoding="utf-8"); display(Markdown("## Report-ready conclusion\n\n"+text)); zf.close()


## Report-ready conclusion

# Quality-weight confirmation

Across three matched seeds, quality weighting changed ROUGE-L by **+0.44 ± 0.98**
and F1 by **+0.45 ± 1.08** relative to the unchanged objective.
The improvement is **not yet supported** by the predeclared practical rule: positive mean and positive ROUGE-L
effect in every seed. Results are validation-only; the locked test set was not accessed.
